# RAG Evaluation Notebook

Evaluates the RAG pipeline against the oracle benchmark dataset (`RAG_test/benchmark_dataset.json`).

**Prerequisites**
- Chunked filings must be cached: run `python -m RAG_test.cache_chunked_filings` first
- `OPENAI_API_KEY` must be set in your environment

In [ ]:
import sys
from pathlib import Path

# Ensure repo root is on sys.path
REPO_ROOT = Path("__file__").resolve().parents[1] if "__file__" in dir() else Path.cwd()
while not (REPO_ROOT / "backend").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")

In [ ]:
from statistics import mean
from datetime import datetime, timezone

from RAG_test.common import (
    BENCHMARK_DATASET_PATH,
    CHUNKED_FILINGS_DIR,
    EMBEDDING_CACHE_DIR,
    RESULTS_DIR,
    chunked_filings_path,
    ensure_data_dirs,
    load_json,
    write_json,
)
from RAG_test.evaluators import (
    answer_matches_oracle,
    evaluate_oracle_match_with_llm,
    evaluate_answer_with_llm,
    optional_bertscore,
)
from backend.agents.retrieval_pipeline import (
    FilingRetrievalPipeline,
    JsonEmbeddingCache,
    OpenAIChatGenerationProvider,
    OpenAIEmbeddingProvider,
    build_chunk_records_from_prepared_filings,
)

ensure_data_dirs()
print("Imports OK")

## Configuration

Adjust the settings below before running the evaluation.

In [ ]:
# --- Settings ---
EMBEDDING_MODEL   = "text-embedding-3-small"
GENERATION_MODEL  = "gpt-4o-mini"
USE_EMBEDDING_CACHE = True   # set False to force fresh embeddings
USE_LLM_JUDGE     = True     # set True to get relevance/completeness/faithfulness scores
TOP_K             = None     # None = use per-question-type recommended_k; or set an int e.g. 5

DATASET_PATH  = BENCHMARK_DATASET_PATH
REPORT_PATH   = RESULTS_DIR / "rag_evaluation_report.json"

## Load Dataset & Initialise Providers

In [ ]:
dataset = load_json(DATASET_PATH)
print(f"Loaded {len(dataset)} benchmark examples")

from collections import Counter
type_counts = Counter(ex["question_type"] for ex in dataset)
for qtype, count in sorted(type_counts.items()):
    print(f"  {qtype}: {count}")

In [ ]:
embedding_cache = None
if USE_EMBEDDING_CACHE:
    cache_file = EMBEDDING_CACHE_DIR / f"{EMBEDDING_MODEL.replace('/', '__')}.json"
    embedding_cache = JsonEmbeddingCache(cache_file)
    print(f"Embedding cache: {cache_file}")

embedding_provider  = OpenAIEmbeddingProvider(model=EMBEDDING_MODEL, cache=embedding_cache)
generation_provider = OpenAIChatGenerationProvider(model=GENERATION_MODEL)
print(f"Embedding model : {EMBEDDING_MODEL}")
print(f"Generation model: {GENERATION_MODEL}")

## Run Evaluation

In [ ]:
def select_filings(cached_payload, filing_date):
    filings = cached_payload.get("filings", [])
    if not filing_date:
        return [max(filings, key=lambda f: f.get("filing_date", ""))] if filings else []
    return [f for f in filings if f.get("filing_date") == filing_date]


pipeline_cache = {}

def get_pipeline(ticker, filing_date):
    key = (ticker, filing_date)
    if key in pipeline_cache:
        return pipeline_cache[key]
    payload  = load_json(chunked_filings_path(ticker))
    filings  = select_filings(payload, filing_date)
    if not filings:
        raise ValueError(f"No cached filings for {ticker} filing_date={filing_date!r}")
    prepared = {"ticker": ticker, "filings": filings}
    chunks   = build_chunk_records_from_prepared_filings(prepared)
    pipeline = FilingRetrievalPipeline(embedding_provider)
    pipeline.index_chunks(chunks)
    pipeline_cache[key] = (pipeline, filings)
    return pipeline_cache[key]

In [ ]:
results = []

for i, example in enumerate(dataset):
    ticker       = example["ticker"]
    filing_date  = example.get("filing_date")
    cache_path   = chunked_filings_path(ticker)

    if not cache_path.exists():
        raise FileNotFoundError(
            f"Missing chunk cache for {ticker}. Run RAG_test/cache_chunked_filings.py first."
        )

    pipeline, filings = get_pipeline(ticker, filing_date)

    answer_result  = pipeline.answer_question(
        question=example["question"],
        generation_provider=generation_provider,
        k=TOP_K,
    )
    retrieved_chunks        = answer_result.get("sources", [])
    answer                  = answer_result.get("answer")
    resolved_filing_date    = filing_date or (filings[0].get("filing_date") if filings else None)

    oracle_answer_judge = evaluate_oracle_match_with_llm(
        generation_provider,
        example["question"],
        answer,
        example.get("oracle_answer"),
    )
    answer_correct = (
        oracle_answer_judge.get("correct")
        if oracle_answer_judge and oracle_answer_judge.get("correct") is not None
        else answer_matches_oracle(answer, example.get("oracle_answer"))
    )

    llm_judge = None
    if USE_LLM_JUDGE:
        llm_judge = evaluate_answer_with_llm(
            generation_provider,
            example["question"],
            answer or "",
            retrieved_chunks,
        )

    results.append({
        **example,
        "resolved_filing_date": resolved_filing_date,
        "generated_answer": answer,
        "retrieved_chunks": retrieved_chunks,
        "answer_correct": answer_correct,
        "oracle_answer_judge": oracle_answer_judge,
        "llm_judge": llm_judge,
    })

    status = "CORRECT" if answer_correct else ("WRONG" if answer_correct is False else "NO ORACLE")
    print(f"[{i+1:02d}/{len(dataset)}] {ticker} | {example['question_type']:<22} | {status}")

print("\nDone.")

## Summary Metrics

In [ ]:
answer_rows = [r for r in results if r["answer_correct"] is not None]
judge_rows  = [r for r in results if r.get("llm_judge")]

accuracy = mean(1.0 if r["answer_correct"] else 0.0 for r in answer_rows) if answer_rows else None

llm_overall = None
if judge_rows:
    overall_scores = [
        r["llm_judge"]["scores"]["overall"]
        for r in judge_rows
        if r["llm_judge"]["scores"]["overall"] is not None
    ]
    llm_overall = mean(overall_scores) if overall_scores else None

print(f"Examples evaluated : {len(results)}")
print(f"Answer accuracy    : {accuracy:.1%}" if accuracy is not None else "Answer accuracy    : N/A")
print(f"LLM judge overall  : {llm_overall:.2f}/5" if llm_overall is not None else "LLM judge overall  : N/A")

In [ ]:
# Per-question-type accuracy
from collections import defaultdict

by_type = defaultdict(list)
for r in answer_rows:
    by_type[r["question_type"]].append(r["answer_correct"])

print(f"{'Question type':<25} {'Correct':>7} {'Total':>6} {'Accuracy':>9}")
print("-" * 52)
for qtype, flags in sorted(by_type.items()):
    correct = sum(flags)
    total   = len(flags)
    print(f"{qtype:<25} {correct:>7} {total:>6} {correct/total:>9.1%}")

In [ ]:
# Per-company accuracy
by_company = defaultdict(list)
for r in answer_rows:
    by_company[r["ticker"]].append(r["answer_correct"])

print(f"{'Ticker':<8} {'Correct':>7} {'Total':>6} {'Accuracy':>9}")
print("-" * 35)
for ticker, flags in sorted(by_company.items()):
    correct = sum(flags)
    total   = len(flags)
    print(f"{ticker:<8} {correct:>7} {total:>6} {correct/total:>9.1%}")

## Inspect Individual Results

In [ ]:
# Show all incorrect answers
wrong = [r for r in results if r["answer_correct"] is False]
print(f"{len(wrong)} incorrect answers\n")

for r in wrong:
    print(f"{'='*80}")
    print(f"Ticker       : {r['ticker']}  |  {r['question_type']}")
    print(f"Question     : {r['question']}")
    print(f"Oracle       : {r.get('oracle_answer', 'N/A')}")
    print(f"Generated    : {r.get('generated_answer', 'N/A')}")
    if r.get("oracle_answer_judge"):
        print(f"Judge reason : {r['oracle_answer_judge'].get('reason', '')}")

In [ ]:
# Show LLM judge scores breakdown (if enabled)
if judge_rows:
    print(f"{'Ticker':<6} {'Type':<22} {'Rel':>4} {'Comp':>5} {'Faith':>6} {'Overall':>8}")
    print("-" * 55)
    for r in results:
        if not r.get("llm_judge"):
            continue
        sc = r["llm_judge"]["scores"]
        print(
            f"{r['ticker']:<6} {r['question_type']:<22} "
            f"{str(sc.get('relevance','?')):>4} "
            f"{str(sc.get('completeness','?')):>5} "
            f"{str(sc.get('faithfulness','?')):>6} "
            f"{str(sc.get('overall','?')):>8}"
        )
else:
    print("LLM judge was disabled. Set USE_LLM_JUDGE = True and re-run.")

## Save Report

In [ ]:
# Optional BERTScore (requires bert-score package)
pipeline_bertscores = optional_bertscore(
    [r.get("generated_answer") or "" for r in answer_rows],
    [r.get("oracle_answer") or "" for r in answer_rows],
)
if pipeline_bertscores:
    for row, score in zip(answer_rows, pipeline_bertscores):
        row["bertscore_f1"] = score
    print(f"BERTScore F1 (mean): {mean(pipeline_bertscores):.4f}")
else:
    print("BERTScore not computed (install bert-score to enable)")

report = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_path": str(DATASET_PATH),
    "chunk_cache_dir": str(CHUNKED_FILINGS_DIR),
    "embedding_model": EMBEDDING_MODEL,
    "generation_model": GENERATION_MODEL,
    "k": TOP_K if TOP_K is not None else "per_question_type",
    "summary": {
        "num_examples": len(results),
        "answer_accuracy": accuracy,
        "pipeline_bertscore_f1": mean(pipeline_bertscores) if pipeline_bertscores else None,
        "llm_judge_overall": llm_overall,
    },
    "results": results,
}

write_json(REPORT_PATH, report)
print(f"Report saved to {REPORT_PATH}")